# Namen skripte

Skripta predstavlja celoten potek priprave in analize PlanetScope satelitskih posnetkov za spremljanje premikov v plazu.

Temelji na članku o tej tematiki: https://doi.org/10.5194/esurf-12-1121-2024

Za poganjanje je potreben Linux/WSL sistem z nameščenim Ames Stereo Pipeline (ASP), skripte iz repozitorija https://github.com/UP-RS-ESP/PlanetScope_landslide_tracking in Planet račun za prenos posnetkov.

V skripti naredimo naslednje korake:
- preverimo in preberemo AOI (območje interesa), ki ga uporabljamo za iskanje PlanetScope posnetkov,
- pošljemo iskalni zahtevek na Planet API in filtriramo posnetke po času, oblačnosti in prekrivanju z AOI,
- izberemo najboljši par posnetkov pred in po dogodku na podlagi kvalitete in geometrijskih pogojev,
- prenesemo ter obrežemo izbrane PlanetScope posnetke na območje AOI,
- pripravimo zeleni kanal za sledenje odmikov pikslov, ki je manj občutljiv na atmosferske motnje,
- izvedemo korelacijo premikov z Ames Stereo Pipeline (ASP),
- popravimo sistematične napake s polinomskim prileganjem in izračunamo hitrost ter velikost premika,
- ter na koncu izračunamo osnovno statistiko premikov znotraj plazovne maske.


## 0. Priprava

In [ ]:
# Knjižnice
import os
import requests
import pandas as pd
import geopandas as gpd
import time
from shapely.geometry import Polygon, mapping, shape
import rasterio 
from rasterio.windows import from_bounds
from pyproj import Transformer
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from datetime import datetime
import sys, platform
from pathlib import Path
import shutil
from rasterio.mask import mask

# ASP funckije
sys.path.append("/home/USER/PlanetScope_landslide_tracking") # pot do skript PlanetScope_landslide_tracking

# V mapi PlanetScope_landslide_tracking je potrebno ustvariti datoteko __init__.py, ki pove, da je mapa Python paket, da lahko uvozimo funkcije.

import asp_helper_functions as asp
import optimization_functions as opt
import postprocessing_functions as postprocessing

In [ ]:
# Planet API ključ - vidite ga tukaj: https://insights.planet.com/account/#/settings
if os.environ.get('PL_API_KEY', ''):
    API_KEY = os.environ.get('PL_API_KEY', '')
else:
    try:
        with open('planet_api_kljuc.txt', 'r') as f:    # v isti mapi kot je ta skripta, ustvarite datoteko planet_api_kljuc.txt, v kateri bo napisan samo vaš API ključ
            API_KEY = f.read().strip()
    except FileNotFoundError:
        raise ValueError("API ključ ni nastavljen. Nastavite okolje spremenljivko PL_API_KEY ali ustvarite datoteko planet_api_key.txt z vašim API ključem.")

In [ ]:
# Izpišimo API ključ, da preverimo, če je bil pravilno naložen
print(f'API ključ naložen: {API_KEY[:4]}...{API_KEY[-4:]}')

In [ ]:
# Nastavite sejo
session = requests.Session()

# Avtentikacija z API ključem
session.auth = (API_KEY, "")

# Nastavi osnovni URL za podatkovni API
URL = "https://api.planet.com/data/v1"

In [ ]:
# Preverimo ali je API ključ veljaven z GET zahtevkom za tip podatkov (item type).
test_url = f'{URL}/item-types'
resp = session.get(test_url)
assert resp.ok, f'Nekaj ni v redu: {resp.status_code}, {resp.content}'
print("Vse je pripravljeno, začnimo!")

## 1. Iskanje posnetkov

### 1.1 Območje interesa

In [ ]:
# Ime datoteke z AOI
aoi_file = "./podatki/Plaz_BB.shp" # . pomeni trenutno mapo; če želite iti en nivo višje, napišite ..

# Pretvorimo v geopandas podatkovni okvir
aoi_gdf = gpd.read_file(aoi_file)

# Najprej preverimo trenutni CRS
print("Trenutni CRS:", aoi_gdf.crs)

# Pretvori v EPSG:4326, če še ni
if aoi_gdf.crs is None:
    raise ValueError("Datoteka nima definiranega CRS. Najprej ga moraš pravilno nastaviti.")
elif aoi_gdf.crs.to_epsg() != 4326:
    aoi_gdf = aoi_gdf.to_crs(epsg=4326)
    print("Spremenil sem CRS v EPSG:4326.")

# Pretvorimo v GeoJSON obliko    
gdf_geom_gjson = aoi_gdf.geometry.iloc[0].__geo_interface__
gdf_geom_gjson

### 1.2 Filtri

In [ ]:
# Nastavimo nekaj filtrov

# prekrivanje z AOI 
geometry_filter = {
  "type": "GeometryFilter",
  "field_name": "geometry",
  "config": gdf_geom_gjson
}

# časovno obdobje
date_range_filter = {
  "type": "DateRangeFilter",
  "field_name": "acquired",
  "config": {
    "gte": "2023-07-15T00:00:00.000Z",  # začetek
    "lte": "2023-08-15T00:00:00.000Z"   # konec
  }
}

# <30 % oblačnost
cloud_cover_filter = {
  "type": "RangeFilter",
  "field_name": "cloud_cover",
  "config": {
    "lte": 0.3
  }
}

###
# združen filter
combined_filter = {
  "type": "AndFilter",
  "config": [geometry_filter, date_range_filter, cloud_cover_filter]
}

In [ ]:
# datum dogodka (plazu)
event_date = "2023-08-04"  # datum plazu v obliki "YYYY-MM-DD"
event_dt = datetime.fromisoformat(event_date)

### 1.3 Iskanje prek http zahtevka

In [ ]:
# Poišče vse posnetke, ki ustrezajo kriterijem (filtrom)
item_type = "PSScene"

# API zahtevek objekt
search_request = {
    "item_types": [item_type],
    "filter": combined_filter
}

# pošlji zahtevo POST
search_result = session.post(
    "https://api.planet.com/data/v1/quick-search",
    json=search_request
)

results = search_result.json()

features = results.get("features", [])

if not features:
    raise ValueError("Ni rezultatov – preveri filter ali AOI.")

# --- pregled datumov ---
dates = [f["properties"]["acquired"][:10] for f in features]
count = Counter(dates)

print(f"\nNa voljo so produkti za {len(count)} datumov:\n")

for d, c in sorted(count.items()):
    print(f"{d} ({c} posnetkov)")

# --- razdelitev pred / po dogodku ---
before = []
after = []

for f in features:
    acquired_dt = datetime.fromisoformat(
        f["properties"]["acquired"].replace("Z", "+00:00")
    ).replace(tzinfo=None)

    if acquired_dt.date() < event_dt.date():
        before.append(f)
    else:
        after.append(f)

def print_scene(f):
    p = f["properties"]
    print(
        f"ID: {f['id']} | "
        f"{p['acquired']} | "
        f"cloud: {p['cloud_cover']:.2f} | "
        f"view: {p.get('view_angle')} | "
        f"azimuth: {p.get('satellite_azimuth')} | "
        f"sat: {p.get('satellite_id')} | "
        f"sun_el: {p.get('sun_elevation')} | "
        f"quality: {p.get('quality_category')}"
    )

print("\nPOSNETKI PRED DOGODKOM:\n")

for f in sorted(before, key=lambda x: x["properties"]["acquired"]):
    print_scene(f)

print("\n" + "-" * 80 + "\n")

print("POSNETKI PO DOGODKU:\n")

for f in sorted(after, key=lambda x: x["properties"]["acquired"]):
    print_scene(f)

In [ ]:
# iskanje najboljšega para pred/po dogodku

max_view_diff = 2      # stopinje
max_azimuth_diff = 10  # stopinje

# obdrži samo standard quality
before_std = [
    f for f in before
    if f["properties"].get("quality_category") == "standard"
]

after_std = [
    f for f in after
    if f["properties"].get("quality_category") == "standard"
]

def azimuth_diff(a1, a2):
    """Pravilna razlika azimuta, ker je azimut krožen 0–360."""
    diff = abs(a1 - a2)
    return min(diff, 360 - diff)

valid_pairs = []

for b in before_std:
    pb = b["properties"]

    for a in after_std:
        pa = a["properties"]

        view_diff = abs(pb["view_angle"] - pa["view_angle"])
        az_diff = azimuth_diff(pb["satellite_azimuth"], pa["satellite_azimuth"])

        if view_diff <= max_view_diff and az_diff <= max_azimuth_diff:
            mean_cloud = (pb["cloud_cover"] + pa["cloud_cover"]) / 2

            valid_pairs.append({
                "before": b,
                "after": a,
                "view_diff": view_diff,
                "azimuth_diff": az_diff,
                "mean_cloud": mean_cloud
            })

if not valid_pairs:
    print("Ni najdenega ustreznega para.")
    print("Spremeni filter: povečaj dovoljeno razliko view angle / azimuth ali razširi časovno okno.")
else:
    # izberi par z najmanjšo povprečno oblačnostjo
    best_pair = min(valid_pairs, key=lambda x: x["mean_cloud"])

    before_feat = best_pair["before"]
    after_feat = best_pair["after"]

    print("Najden najboljši par:\n")

    print("PRED:")
    print("ID:", before_feat["id"])
    print("Datum:", before_feat["properties"]["acquired"])
    print("Cloud:", before_feat["properties"]["cloud_cover"])
    print("View angle:", before_feat["properties"]["view_angle"])
    print("Satellite azimuth:", before_feat["properties"]["satellite_azimuth"])
    print()

    print("PO:")
    print("ID:", after_feat["id"])
    print("Datum:", after_feat["properties"]["acquired"])
    print("Cloud:", after_feat["properties"]["cloud_cover"])
    print("View angle:", after_feat["properties"]["view_angle"])
    print("Satellite azimuth:", after_feat["properties"]["satellite_azimuth"])
    print()

    print("RAZLIKE:")
    print(f"View angle diff: {best_pair['view_diff']:.1f}")
    print(f"Azimuth diff: {best_pair['azimuth_diff']:.1f}")
    print(f"Povprečna oblačnost: {best_pair['mean_cloud']:.1f}")

### 1.4 Prenos posnetkov

In [ ]:
# Vrsta produkta, ki ga želimo prenesti
asset = "ortho_analytic_4b_sr"

# Ustvarimo mapo za rezultate, če še ne obstaja
out_dir = Path(".") / "rezultati"
out_dir.mkdir(exist_ok=True)

In [ ]:
"""
    Funkcija za prenos in pripravo posnetka:
    1. preveri in po potrebi aktivira izbran Planet asset,
    2. iz COG prebere samo del posnetka, ki pokriva AOI,
    3. shrani obrezan večkanalni TIFF,
    4. iz njega izloči zeleni kanal za sledenje odmikov pikslov.
"""

def download_and_prepare_scene(feat, label):
    # Planet feature vsebuje ID posnetka in footprint/geometrijo scene.
    item_id = feat["id"]
    item_geom = feat["geometry"]

    # Preveri, ali je izbrani asset za ta posnetek sploh na voljo. 
    if asset not in feat["assets"]:
        raise ValueError(f"Asset '{asset}' is not available for scene {item_id}.")

    # URL, prek katerega dobimo status asseta in povezave za aktivacijo/prenos.
    assets_url = f"{URL}/item-types/{item_type}/items/{item_id}/assets"

    # Pridobi trenutne informacije o assetu.
    asset_info = session.get(assets_url).json()[asset]

    # Planet asset mora biti aktiven, preden lahko dobimo download link.
    if asset_info["status"] != "active":
        # Pošlji zahtevo za aktivacijo asseta.
        activation_link = asset_info["_links"]["activate"]
        session.get(activation_link)

        # Aktivacija je asinhrona, zato čakamo in redno preverjamo status.
        while session.get(assets_url).json()[asset]["status"] != "active":
            print(f"{label}: čakam aktivacijo...")
            time.sleep(30)

    # Ko je asset aktiven, Planet vrne začasno povezavo do COG datoteke.
    download_link = session.get(assets_url).json()[asset]["location"]

    # Pretvori AOI in footprint scene v Shapely geometrije.
    aoi_shape = shape(gdf_geom_gjson)
    item_shape = shape(item_geom)
    # Uporabi samo del AOI-ja, ki se dejansko prekriva s posnetkom.
    clipped_shape = item_shape.intersection(aoi_shape)

    # Imeni izhodnih datotek. Label pove, ali gre za before ali after posnetek.
    clipped_tif = out_dir / f"{item_id}_cl_{label}.tif"     # ne spreminjaj imena, ker drugače naprej ne dela
    green_tif = out_dir / f"{item_id}_3B_green_{label}.tif" # ne spreminjaj imena, ker drugače naprej ne dela

    # Odpri oddaljeni COG prek download linka; celotne scene ne prenesemo vnaprej.
    with rasterio.open(download_link) as src:
        # AOI je v EPSG:4326, raster pa ima lahko drug CRS, zato AOI transformiramo v CRS rasterja.
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)

        aoi_transformed = Polygon([
            transformer.transform(*coords)
            for coords in mapping(clipped_shape)["coordinates"][0]
        ])

        # Izračunaj raster okno, ki zajame transformirani AOI.
        window = from_bounds(*aoi_transformed.bounds, transform=src.transform)

        # Preberi samo to okno in izračunaj novo georeferenciranje za izrez.
        clipped_data = src.read(window=window)
        clipped_transform = src.window_transform(window)

        # Kopiraj metapodatke iz originala in jih prilagodi velikosti izreza.
        out_meta = src.meta.copy()
        out_meta.update({
            "height": clipped_data.shape[1],
            "width": clipped_data.shape[2],
            "transform": clipped_transform,
            "driver": "COG"
        })

        # Shrani obrezan večkanalni posnetek.
        with rasterio.open(clipped_tif, "w", **out_meta) as dst:
            dst.write(clipped_data)

    # Iz obrezanega večkanalnega TIFF-a preberi samo zeleni kanal.
    with rasterio.open(clipped_tif) as src:
        # PlanetScope vrstni red tukaj: band 1 = blue, band 2 = green, band 3 = red, band 4 = NIR.
        green = src.read(2)

        # Za enokanalni TIFF mora biti count = 1.
        meta = src.meta.copy()
        meta.update({
            "count": 1,
            "driver": "GTiff"
        })

        with rasterio.open(green_tif, "w", **meta) as dst:
            dst.write(green, 1)

    # Vrni vse poti in ID-je, ki jih naslednje celice potrebujejo.
    return {
        "label": label,
        "item_id": item_id,
        "clipped_tif": clipped_tif,
        "green_tif": green_tif,
        "download_link": download_link
    }

Spodnja celica bo prenesla posnetke. Proces lahko traja nekaj minut.

In [ ]:
# Prenos obeh posnetkov s funkcijo download_and_prepare_scene
before_result = download_and_prepare_scene(before_feat, "before")
after_result = download_and_prepare_scene(after_feat, "after")

before_green = before_result["green_tif"]
after_green = after_result["green_tif"]

print("Zelen kanal posnetka pred dogodkom shranjen tukaj:", before_green)
print("Zelen kanal posnetka po dogodku shranjen tukaj:", after_green)

In [ ]:
# Preverim kvoto
url = "https://api.planet.com/auth/v1/experimental/public/my/subscriptions"

res = session.get(url)
subs = res.json()

for s in subs:

    quota = s.get("quota_sqkm")

    if quota is None or quota <= 0:
        continue

    print("PLAN:", s["plan"]["name"])
    print("AKTIVEN:", s["active_from"], "→", s["active_to"])
    print("KVOTA (km²):", s["quota_sqkm"])
    print(f"PORABLJENO (km²): {s['quota_used']:.2f}")
    print(f"OSTANE (km²): {s['quota_sqkm'] - s['quota_used']:.2f}")

### 1.5 Prikaz prenesenih posnetkov

Prikaz RGB posnetkov pred in po dogodku in posebej zelenega kanala, ki ga uporabimo za sledenje odmikov pikslov.

In [ ]:
def stretch_for_display(arr, plow=2, phigh=98):
    """Razteg vrednosti za lepši prikaz rastra."""
    arr = arr.astype(float)
    valid = np.isfinite(arr)

    if not valid.any():
        return arr

    vmin, vmax = np.nanpercentile(arr[valid], (plow, phigh))

    if vmax == vmin:
        return np.zeros_like(arr, dtype=float)

    arr = (arr - vmin) / (vmax - vmin)
    return np.clip(arr, 0, 1)


def read_rgb_for_display(tif_path):
    """Prebere PlanetScope RGB prikaz iz 4-kanalnega TIFF-a.

    Pri tem assetu je vrstni red kanalov: 1 = blue, 2 = green, 3 = red, 4 = NIR.
    Za RGB zato uporabimo bands 3, 2, 1.
    """
    with rasterio.open(tif_path) as src:
        red = stretch_for_display(src.read(3))
        green = stretch_for_display(src.read(2))
        blue = stretch_for_display(src.read(1))

    return np.dstack([red, green, blue])


def read_green_for_display(tif_path):
    """Prebere enokanalni zeleni TIFF in ga pripravi za prikaz."""
    with rasterio.open(tif_path) as src:
        green = src.read(1)

    return stretch_for_display(green)


fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(read_rgb_for_display(before_result["clipped_tif"]))
axes[0, 0].set_title("Pred - RGB")
axes[0, 0].axis("off")

axes[0, 1].imshow(read_rgb_for_display(after_result["clipped_tif"]))
axes[0, 1].set_title("Po - RGB")
axes[0, 1].axis("off")

axes[1, 0].imshow(read_green_for_display(before_result["green_tif"]), cmap="gray")
axes[1, 0].set_title("Pred - zelen kanal")
axes[1, 0].axis("off")

axes[1, 1].imshow(read_green_for_display(after_result["green_tif"]), cmap="gray")
axes[1, 1].set_title("Po - zelen kanal")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()


## 2. Sledenje odmikov pikslov

**Pomembno:** ta del bo deloval šele v Ubuntu/WSL okolju z nameščenim programom Ames Stereo Pipeline (ASP).

Za sledenje odmikov pikslov potrebujemo samo en kanal. Običajno se izbere zelen kanal, ker nanj atmosfera najmanj vpliva.


### 2.1 Preverimo okolje


In [ ]:
print("Python:", sys.executable)
print("OS:", platform.platform())
print("Mapa:", Path.cwd())
print("Linux/WSL:", platform.system() == "Linux")

Nastavi pot do Ames Stereo Pipeline. Popravi `amespath`, da kaže na mapo `bin` tvoje ASP namestitve.

In [ ]:
amespath = "/home/USER/StereoPipeline-3.6.0-2025-12-26-x86_64-Linux/bin"    # pot do nameščenega ASP
parallel_stereo = Path(amespath) / "parallel_stereo"
print("parallel_stereo obstaja:", parallel_stereo.exists())
print("parallel_stereo v PATH:", shutil.which("parallel_stereo"))

### 2.2 Pripravi par za korelacijo

In [ ]:
matches = pd.DataFrame({
    "ref": [str(before_green)],
    "sec": [str(after_green)],
    "id_ref": [before_result["item_id"]],
    "id_sec": [after_result["item_id"]],
    "path": [str(out_dir)]
})

matchfile = out_dir / "matches_manual.csv"  # v tej datoteki bo shranjen seznam parov za ASP
matches.to_csv(matchfile, index=False)

print("Par za sledenje odmikov pikslov:")
print(f"  before ID: {before_result['item_id']}")
print(f"  after ID:  {after_result['item_id']}")
print(f"  before tif: {before_green}")
print(f"  after tif:  {after_green}")
print(f"  matchfile:  {matchfile}")

### 2.3 Korelacija z ASP

Funckija `asp.correlate_asp_wrapper(...)` za vsak piksel oz. okno poišče, kam se je enak vzorec iz prvega posnetka (pred dogodkom) premaknil v drugem posnetku. Rezultat je raster premikov. 

Rezultat se shrani v *.tif datoteko, ki je poimenovana kot: `[id_img1]_[id_img2][prefix_ext]-F.tif`. Datoteka ima 3 kanale, kjer 1. kanal predstavlja premike v smeri vzhod-zahod (dx), 2. kanal predstavlja premike v smeri sever-jug (dy) in 3. kanal masko dobro koreliranih pikslov.


In [ ]:
dmaps = asp.correlate_asp_wrapper(
    amespath,
    matches,
    sp_mode=2, # subpixel mode
    corr_kernel=15, # correlation kernel size
    prefix_ext="_L3B",
    overwrite=True
)

print("Disparity maps:")
for d in dmaps:
    print(d)

#### 2.3.1 Prikaz rezultatov korelacije

In [ ]:
def value_range(*arrays):
    """Vrne skupni dejanski min/max razpon vrednosti za prikaz več rasterjev."""
    vals = []

    for arr in arrays:
        arr = arr.astype(float)
        arr[~np.isfinite(arr)] = np.nan
        vals.append(arr)

    stacked = np.concatenate([arr.ravel() for arr in vals])
    vmin = np.nanmin(stacked)
    vmax = np.nanmax(stacked)

    if not np.isfinite(vmin) or not np.isfinite(vmax):
        return 0, 1

    if vmin == vmax:
        return vmin - 1, vmax + 1

    return vmin, vmax

for disp_map in dmaps:
    disp_map = Path(disp_map)
    
    with rasterio.open(disp_map) as src:
        dx = src.read(1).astype(float)
        dy = src.read(2).astype(float)
        good_pixels = src.read(3) if src.count >= 3 else None

    if good_pixels is not None:
        dx[good_pixels == 0] = np.nan
        dy[good_pixels == 0] = np.nan

    disp_min, disp_max = value_range(dx, dy)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    im0 = axes[0].imshow(dx, cmap="coolwarm", vmin=disp_min, vmax=disp_max)
    axes[0].set_title("Kanal 1: dx")
    axes[0].axis("off")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(dy, cmap="coolwarm", vmin=disp_min, vmax=disp_max)
    axes[1].set_title("Kanal 2: dy")
    axes[1].axis("off")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    if good_pixels is not None:
        im2 = axes[2].imshow(good_pixels, cmap="gray", vmin=0, vmax=1)
        axes[2].set_title("Kanal 3: maska dobro koreliranih pikslov")
        plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04, ticks=[0, 1])
    else:
        axes[2].text(0.5, 0.5, "Ni tretjega kanala", ha="center", va="center")
        axes[2].set_title("Kanal 3")

    axes[2].axis("off")
    plt.tight_layout()
    plt.show()


### 2.4 Polynomial fit
Po korelaciji raster premikov še vedno ne predstavlja samo dejanskega premika površja. V njem so lahko prisotni tudi sistematični napačni signali, ki nastanejo zaradi razlik v geometriji posnetkov, poravnave posnetkov, napak v ortorektifikaciji ...

Zato uporabimo polynomial fit, s katerim modeliramo gladek sistematičen premik po sliki in ga odštejemo od rezultatov korelacije. Prilegamo lahko polinom 1., 2. ali 3. reda. 

Preostale napake lahko dodatno modeliramo s svojim DMR, ki mora biti kar se da aktualen.

Ime datoteke: `[id_img1]_[id_img2][prefix_ext]_polyfit-F.tif`. Ne vsebuje več kanala, ki predstavlja masko dobro koreliranih pikslov, ker je bila ta že uporabljena za oceno dx in dy premikov. 

In [ ]:
dmaps_pfit = opt.apply_polyfit(matches, prefix_ext="_L3B", order=2)

print("Polyfit rezultati:")
for d in dmaps_pfit:
    print(d)

### 2.5 Izračun hitrosti premika, velikosti premika in smeri premika
#### 2.5.1 Hitrost premika [m/leto]
Iz popravljenega rastra premikov (polyfit raster) izračunamo hitrost premika na leto [m/leto]. Rezultat je shranjen v datoteko `[id_img1]_[id_img2][prefix_ext]_polyfit-F_velocity.tif`. Ima 2 kanala:  

1. kanal = hitrost [m/leto]  
2. kanal = smer premika [stopinje]

In [ ]:
postprocessing.calc_velocity_wrapper(
    matches,
    prefix_ext="_L3B_polyfit",
    overwrite=True
)

ref_id = before_result["item_id"]
sec_id = after_result["item_id"]

vel_file = (
    Path("../rezultati/disparity_maps")
    / f"{ref_id}_{sec_id}_L3B_polyfit-F_velocity.tif"
)

#### 2.5.2 Velikost premika [m]

In [ ]:
# Računam iz hitrosti premika
ref_id = before_result["item_id"]
sec_id = after_result["item_id"]

velocity_tif = (
    Path("./rezultati/disparity_maps")
    / f"{ref_id}_{sec_id}_L3B_polyfit-F_velocity.tif"
)

displacement_tif = (
    Path("./rezultati/disparity_maps")
    / f"{ref_id}_{sec_id}_L3B_polyfit-F_displacement.tif"
)

with rasterio.open(velocity_tif) as src:
    velocity = src.read(1).astype(float)
    nodata = src.nodata
    meta = src.meta.copy()

if nodata is not None:
    velocity[velocity == nodata] = np.nan

date_ref = datetime.strptime(ref_id[:15], "%Y%m%d_%H%M%S")
date_sec = datetime.strptime(sec_id[:15], "%Y%m%d_%H%M%S")

dt_days = (date_sec - date_ref).total_seconds() / 86400

print(f"Časovni razmik: {dt_days:.0f} dni")

# velocity je v m/leto, zato jo pretvorimo nazaj v premik v metrih
displacement = velocity * dt_days / 365 # premik [m] = hitrost [m/leto] × časovni razmik [dni] / 365

# Shrani premik kot enokanalni TIFF
meta.update({
    "count": 1,
    "dtype": "float32",
    "nodata": -9999
})

displacement_out = displacement.astype("float32")
displacement_out[np.isnan(displacement_out)] = -9999

with rasterio.open(displacement_tif, "w", **meta) as dst:
    dst.write(displacement_out, 1)

print("Raster premikov shranjen:")
print(displacement_tif)

#### 2.5.3 Prikaz hitrosti in smeri premika ter velikost premika

In [ ]:
# Hitrost in smer
ref_id = before_result["item_id"]
sec_id = after_result["item_id"]

velocity_tif = (
    Path("./rezultati/disparity_maps")
    / f"{ref_id}_{sec_id}_L3B_polyfit-F_velocity.tif"
)

with rasterio.open(velocity_tif) as src:
    velocity = src.read(1).astype(float)
    direction = src.read(2).astype(float)
    nodata = src.nodata

plt.figure(figsize=(8, 6))
plt.imshow(velocity)
plt.colorbar(label="hitrost [m/leto]")
plt.title("Letna hitrost premika")
plt.axis("off")
plt.show()

plt.figure(figsize=(8, 6))
im = plt.imshow(direction, cmap="twilight", vmin=0, vmax=360)
cbar = plt.colorbar(im, label="smer premika")
cbar.set_ticks([0, 90, 180, 270, 360])
cbar.set_ticklabels(["N / 0°", "E / 90°", "S / 180°", "W / 270°", "N / 360°"])
plt.title("Smer premika")
plt.axis("off")
plt.show()


In [ ]:
# velikost premika
plt.figure(figsize=(8, 6))
plt.imshow(displacement)
plt.colorbar(label="Premik[m]")
plt.title("Premik med posnetkoma")
plt.axis("off")
plt.show()

### 2.6 Statistika
Izračun statistike je možen z ASP funckijami (npr. `get_stats_in_aoi`, ki se nahaja v `postprocessing_functions.py`), vendar rezultati predstavljajo letne hitrosti premikov. Ker v tej nalogi obravnavamo enkratne dogodke, tak izračun ni smiseln. Zato bomo uporabili statistike iz knjižnice NumPy, katere funkcije lahko ignorirajo NaN vrednosti, ki se pri nas pojavijo zaradi slabe korelacije med posnetkoma.


In [ ]:
# Priprava maske plazu
maska_plazu_shp = "./podatki/Plaz.shp"
maska_plazu_gdf = gpd.read_file(maska_plazu_shp)

In [ ]:
# Izračunam statistiko premika znotraj maske plazu
with rasterio.open(displacement_tif) as src:
    maska_plazu_raster_crs = maska_plazu_gdf.to_crs(src.crs)

    clipped, _ = mask(
        src,
        maska_plazu_raster_crs.geometry,
        crop=True
    )

    displacement_masked = clipped[0].astype(float)

    if src.nodata is not None:
        displacement_masked[displacement_masked == src.nodata] = np.nan

stats_displacement = {
    "mean_m": round(np.nanmean(displacement_masked), 2),
    "median_m": round(np.nanmedian(displacement_masked), 2),
    "std_m": round(np.nanstd(displacement_masked), 2),
    "min_m": round(np.nanmin(displacement_masked), 2),
    "max_m": round(np.nanmax(displacement_masked), 2),
    "p25_m": round(np.nanpercentile(displacement_masked, 25), 2),
    "p75_m": round(np.nanpercentile(displacement_masked, 75), 2),
    "p95_m": round(np.nanpercentile(displacement_masked, 95), 2),
}

# Zapišem rezultate v tabeli z DataFrame
stats_displacement_df = pd.DataFrame([stats_displacement])
stats_displacement_df


Rezultati:

- **mean_m**: Povprečna vrednost premika.
- **median_m**: Mediana premika.
- **std_m**: Standardni odklon premika (mera razpršenosti okoli povprečja).
- **min_m**: Najmanjša vrednost premika.
- **max_m**: Največja vrednost premika.
- **p25_m**: 25. percentil (25 % vrednosti je manjše od te vrednosti).
- **p75_m**: 75. percentil (75 % vrednosti je manjše od te vrednosti).
- **p95_m**: 95. percentil (95 % vrednosti je manjše od te vrednosti).

Vse vrednosti so v metrih.